# Stage 05 — Data Storage

**Focus:** reproducible save/load; env-driven paths; CSV vs Parquet; raw vs processed.

_Brief cloud note_: S3/data lakes support large-scale storage and efficient querying (partitioning, column pruning). No deep dive today.

In [ ]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install numpy
# !pip install pandas
# !pip install pyarrow
# !pip install python-dotenv

In [1]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: /Users/jaquelyntl/Desktop/Bootcamp/class note

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [OK ]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

All needed files present.


In [2]:
import os, pathlib, datetime as dt
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
RAW_DIR = pathlib.Path(os.getenv("DATA_DIR_RAW", "data/raw"))
PROC_DIR = pathlib.Path(os.getenv("DATA_DIR_PROCESSED", "data/processed"))
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)
print("RAW_DIR:", RAW_DIR.resolve())
print("PROC_DIR:", PROC_DIR.resolve())

RAW_DIR: /Users/jaquelyntl/Desktop/Bootcamp/class note/data/raw
PROC_DIR: /Users/jaquelyntl/Desktop/Bootcamp/class note/data/processed


## Create a sample DataFrame

In [ ]:
import numpy as np

# Seed the generator so this notebook produces the SAME numbers on every run -
# on the projector, and on your machine at home. Without this line the prices
# change each time you run the cell, and so does every file you save from it.
np.random.seed(5) #ensure the random numbers use below are the same every time 

dates = pd.date_range("2024-01-01", periods=10, freq="D")
df = pd.DataFrame({
    'date': dates,
    'ticker': ['AAPL']*10,
    'price': 150 + np.random.randn(10).cumsum()
})
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    10 non-null     datetime64[us]
 1   ticker  10 non-null     str           
 2   price   10 non-null     float64       
dtypes: datetime64[us](1), float64(1), str(1)
memory usage: 372.0 bytes


In [4]:
df.tail(5)

,date,ticker,price
5,2024-01-06,AAPL,153.981127
6,2024-01-07,AAPL,153.071895
7,2024-01-08,AAPL,152.480258
8,2024-01-09,AAPL,152.667862
9,2024-01-10,AAPL,152.337992


### Aside — `try` / `except` / `finally`

A short detour from storage, because several of the saves further down this
notebook are wrapped in one of these — every Parquet write, for instance, because
it depends on an engine that may not be installed.

The function below calls `pront("hello")` — a deliberate typo, so Python raises
`NameError`. Follow the order things happen in:

1. `print("bye")` runs, and `x` becomes `[1, 2, 3]`.
2. `my_func()` raises. The `except` catches it and prints `Done!`.
3. `finally:` runs **whether or not** anything went wrong, and resets `x` to `[]`.

So the last line of the cell shows `[]`, not `[1, 2, 3]`. That is the whole point of
`finally`: it is where cleanup goes when it has to happen either way — closing a
file, releasing a connection, deleting a half-written output.

In [ ]:
def my_func():
    try:
        pront("hello")
    except Exception as e:
        print("There was an error")
        raise e #?

x=[4,5]
try:
    print("bye")
    x+=[1,2,3]
    my_func()
except Exception as e:
    print("Done!")
finally:
    x=[]
    
x    

bye
There was an error
Done!


[]

## Save to CSV (raw) and Parquet (processed)

In [8]:
def ts():
    return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

csv_path = RAW_DIR / f"prices_{ts()}.csv"
df.to_csv(csv_path, index=False)
print("Saved CSV →", csv_path)

parq_path = PROC_DIR / f"prices_{ts()}.parquet"
try:
    df.to_parquet(parq_path)  # uses installed engine if available
    print("Saved Parquet →", parq_path)
except Exception as e:
    print("Parquet save failed (engine missing?). Skipping Parquet demo.")
    print("Error:", e)

Saved CSV → data/raw/prices_20260814-123130.csv
Parquet save failed (engine missing?). Skipping Parquet demo.
Error: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.


### Quick aside — `all()` and list comprehensions

The validation step just below uses `all(...)` over a list comprehension. Before we get
back to saving and reloading, here is what that construct actually does.


In [9]:
['apple']*2

['apple', 'apple']

In [ ]:
#all() check if all element is 1/True
all([1,1,1])

True

In [16]:
[c == 'price' for c in ['col1', 'col2']]

[False, False]

In [12]:
all([c in ['col1', 'col2'] for c in ['col1', 'col2']])

True

In [13]:
# `[value] * n` builds a list by REPEATING an item - a quick way to make test data.
lst = [True] * 3
print(lst)            # [True, True, True]

# `all(...)` is True only when EVERY element is truthy.
print(all(lst))                                          # True  - all three are True
print(all([1, 1, 1, 1, 1]))                              # True  - every element is non-zero

# A list comprehension builds the list of True/False first, then all() checks it.
print(all([c in ['a', 'b'] for c in ['a', 'b', 'c']]))   # False - 'c' is not in ['a', 'b']

# The same logic written out as an explicit loop:
my_list = []
for c in ['a', 'b', 'c']:
    my_list.append(c in ['a', 'b'])

print(my_list)        # [True, True, False]
print(all(my_list))   # False


[True, True, True]
True
True
False
[True, True, False]
False


In [17]:
df.columns

Index(['date', 'ticker', 'price'], dtype='str')

## Reload & Validate

In [ ]:
def validate_loaded(original: pd.DataFrame, reloaded: pd.DataFrame, cols=('date','ticker','price')):
    checks = {
        'shape_equal': original.shape == reloaded.shape,
        'cols_present': all(c in reloaded.columns for c in cols)
    }
    # dtype sanity checks
    if 'price' in reloaded.columns:
        checks['price_is_numeric'] = pd.api.types.is_numeric_dtype(reloaded['price'])
    if 'date' in reloaded.columns:
        checks['date_is_datetime'] = pd.api.types.is_datetime64_any_dtype(reloaded['date'])
    return checks

df_csv = pd.read_csv(csv_path, parse_dates=['date']) # parse_date is a parameter for read_csv, which turns the designated column to datetime type 
print('CSV validation:', validate_loaded(df, df_csv))

if parq_path.exists():
    try:
        df_parq = pd.read_parquet(parq_path)
        print('Parquet validation:', validate_loaded(df, df_parq))
    except Exception as e:
        print('Parquet read failed:', e)
else:
    print('Parquet file not present (skipped earlier).')

CSV validation: {'shape_equal': True, 'cols_present': True, 'price_is_numeric': True, 'date_is_datetime': True}
Parquet file not present (skipped earlier).


## IO Utilities (suffix-based)

In [ ]:
from typing import Union

def ensure_dir(path: pathlib.Path):
    path.parent.mkdir(parents=True, exist_ok=True)

def detect_format(path: Union[str, pathlib.Path]):
    suf = str(path).lower()
    if suf.endswith('.csv'): return 'csv'
    if suf.endswith('.parquet') or suf.endswith('.pq') or suf.endswith('.parq'): return 'parquet'
    raise ValueError('Unsupported format for: ' + str(path))

def write_df(df: pd.DataFrame, path: Union[str, pathlib.Path]):
    path = pathlib.Path(path)
    ensure_dir(path)
    fmt = detect_format(path)
    if fmt == 'csv':
        df.to_csv(path, index=False)
    elif fmt == 'parquet':
        try:
            df.to_parquet(path)
        except Exception as e:
            raise RuntimeError('Parquet engine not available. Install pyarrow or fastparquet.') from e
    return path

def read_df(path: Union[str, pathlib.Path]):
    path = pathlib.Path(path)
    fmt = detect_format(path)
    if fmt == 'csv':
        return pd.read_csv(path, parse_dates=['date']) if 'date' in pd.read_csv(path, nrows=0).columns else pd.read_csv(path)
    elif fmt == 'parquet':
        try:
            return pd.read_parquet(path)
        except Exception as e:
            raise RuntimeError('Parquet engine not available. Install pyarrow or fastparquet.') from e

# Demo utility usage
csv2 = RAW_DIR / f"prices_util_{ts()}.csv"
pq2  = PROC_DIR / f"prices_util_{ts()}.parquet"
write_df(df, csv2)
df2 = read_df(csv2)
print('Reloaded CSV via util, shape:', df2.shape)

try:
    write_df(df, pq2)
    df3 = read_df(pq2)
    print('Reloaded Parquet via util, shape:', df3.shape)
except RuntimeError as e:
    print('Parquet util demo skipped:', e)

## Non-structured data example: JSON with nested/unstructured data

Here we demonstrate a type of data that **cannot be stored directly as clean structured tables** (CSV or Parquet) without losing information (column types, for instance — further down, CSV turns the nested dict into plain text, and Parquet hands the inner list back as a numpy array rather than a list) or changing format drastically.

This data is a list of records with nested notes and metadata fields, simulating something like JSON logs or documents.

We save it as a JSON file, showing how non-tabular data often requires different storage formats.

In [ ]:
import json

unstructured_data = [
    {
        "date": "2024-01-01",
        "ticker": "AAPL",
        "price": 150.0,
        "notes": {
            "summary": "Stable",
            "details": ["No major events", "Market steady"]
        }
    },
    {
        "date": "2024-01-02",
        "ticker": "AAPL",
        "price": 151.2,
        "notes": {
            "summary": "Slight increase",
            "details": ["Earnings beat expectations", "Analyst upgrades"]
        }
    }
]

json_path = RAW_DIR / f"unstructured_{ts()}.json"
with open(json_path, 'w') as f:
    json.dump(unstructured_data, f, indent=2)

print(f"Saved unstructured JSON data → {json_path}") # Print the path where the JSON data was saved

Saved unstructured JSON data → data/raw/unstructured_20260814-125635.json


## Can this data go straight into CSV or Parquet?

- The `notes` field contains nested dicts and lists — complex hierarchical data.
- CSV supports only flat tabular data — it can't store nested structures without flattening or serialization hacks.
- Parquet supports nested types but converting arbitrarily complex nested JSON with mixed types and lists is non-trivial and often lossy or requires schema design.

Let's try to naively convert this into a DataFrame and save to CSV or Parquet to see what happens.

In [ ]:
# Naive flattening: convert to DataFrame directly
df_unstructured = pd.DataFrame(unstructured_data)
df_unstructured.info()
print(df_unstructured)

# Try saving to CSV
try:
    csv_unstructured_path = RAW_DIR / f"unstructured_naive_{ts()}.csv"
    df_unstructured.to_csv(csv_unstructured_path, index=False)
    print(f"Naively saved unstructured data as CSV → {csv_unstructured_path}")
except Exception as e:
    print("Failed to save unstructured data as CSV:", e)

# Try saving to Parquet
try:
    parq_unstructured_path = PROC_DIR / f"unstructured_naive_{ts()}.parquet"
    df_unstructured.to_parquet(parq_unstructured_path)
    print(f"Naively saved unstructured data as Parquet → {parq_unstructured_path}")
except Exception as e:
    print("Failed to save unstructured data as Parquet:", e)

### Both saves "worked" — so what is the problem?

Neither save raised. No error, no warning, two success messages. **That is the
trap.** The question was never *"did a file get written?"* — it is *"is the data
still there when you read it back?"*

Before running the next cell: `notes` is a dictionary containing a list. Which of
these two formats do you think can hold that, and which one cannot?

In [ ]:
# The saves "succeeded". Reload both and check what actually survived.
back_csv = read_df(csv_unstructured_path)
have_pq  = parq_unstructured_path.exists()      # skip gracefully if no engine
back_pq  = read_df(parq_unstructured_path) if have_pq else None

print("original      notes[0] is a", type(unstructured_data[0]['notes']).__name__)
print("from CSV      notes[0] is a", type(back_csv['notes'][0]).__name__)
if have_pq:
    print("from Parquet  notes[0] is a", type(back_pq['notes'][0]).__name__)

# CSV has no type for a dict, so it wrote the dict's PRINTED FORM. Ask it anyway:
try:
    print("\nCSV     - ask it for the summary:", back_csv['notes'][0]['summary'])
except TypeError as e:
    print("\nCSV     - ask it for the summary:", type(e).__name__ + ":", e)
    print("          what came back was text:", repr(back_csv['notes'][0])[:64])

# Parquet kept the nesting, so the original question still has an answer:
if have_pq:
    print("Parquet - ask it for the summary:", back_pq['notes'][0]['summary'])
else:
    print("Parquet - not written (engine missing?), skipping this half.")

# ...but "survived" is not the same as "identical". Look one level closer:
if have_pq:
    was = unstructured_data[0]['notes']['details']
    now = back_pq['notes'][0]['details']
    print("\ndetails went in as a", type(was).__name__,
          "and came back as a", type(now).__name__,
          "- same values:", list(now) == list(was))
    try:
        print("   is the whole dict equal?",
              back_pq['notes'][0] == unstructured_data[0]['notes'])
    except ValueError as e:
        print("   is the whole dict equal? ValueError:", e)
        print("   comparing an array elementwise gives an array, not one True/False.")

print("\nCSV is a flat text format - it flattened the dictionary and told us nothing.")
print("Parquet kept the structure and the values, but not the exact Python types:")
print("the inner list came back as a numpy array. Close enough to work with, and")
print("NOT close enough for a naive == check - so verify a round trip, never assume")
print("one. Deeply nested or mixed-type data is harder still, which is why the raw")
print("JSON stays in data/raw/ as the source of truth: you can always re-derive a")
print("table, but you cannot re-derive structure you already threw away.")

## Summary
- Env-driven paths → portable IO.
- CSV vs Parquet → tradeoffs.
- Utilities abstract away format details.
- Non-structured data often requires different storage formats (like JSON files).
- Parquet handles nested data to some extent, but deeply nested or mixed data is tricky.
- Next: document storage plan in README.